<h1>Lab 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>

## What You Will Learn

In this lab, you will understand how language models process text by breaking it down into tokens and converting them into numerical representations called embeddings. Think of this as learning the "language" that allows computers to understand and work with human text.

### Key Concepts:
- **Tokens**: The smallest units of text that a model processes (could be words, parts of words, or characters)
- **Embeddings**: Numerical vectors that capture the meaning of tokens in a way computers can understand
- **Why this matters**: Every modern language model - from ChatGPT to BERT - uses these fundamental building blocks

## Setup Instructions

💡 **IMPORTANT**: We need a GPU to run these examples efficiently. In Google Colab:
1. Go to **Runtime > Change runtime type**
2. Set **Hardware accelerator** to **GPU**
3. Set **GPU type** to **T4**

**Why do we need a GPU?** Language models are large neural networks with millions of parameters. GPUs can perform the mathematical operations needed for these models much faster than regular CPUs - often 10-100x faster!

In [ ]:
# Install required libraries
# This might take a few minutes - be patient!
!pip install --upgrade transformers sentence-transformers gensim scikit-learn accelerate peft

# Part 1: Downloading and Running a Language Model

## What's happening here?

We're going to download a real language model and watch it generate text. But first, let's understand the two key components:

### 1. The Tokenizer
Think of the tokenizer as a translator that converts human-readable text into a sequence of numbers (token IDs) that the model can process. It's like breaking a sentence into LEGO blocks that the model can work with.

### 2. The Model
The actual neural network that takes these token IDs and generates predictions for what should come next. It's trained on billions of words and has learned patterns in language.

**Why keep them separate?** We often want to analyze tokenization independently from the model's predictions, which you'll see in the examples below.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model - this downloads ~7GB, so it takes a few minutes
# We're using Microsoft's Phi-3, a smaller but capable language model
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",  # Use the GPU
    torch_dtype="auto",  # Use optimal data type for speed
    trust_remote_code=False,
)

# Load the tokenizer separately
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

## Let's Generate Some Text!

Now we'll see the model in action. Here's what happens step-by-step:

1. **Input**: We provide a text prompt
2. **Tokenization**: The tokenizer converts text → token IDs
3. **Generation**: The model predicts what tokens should come next
4. **Decoding**: Token IDs are converted back → readable text

Notice we're limiting output to 20 tokens (`max_new_tokens=20`) to keep it short for demonstration.

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Step 1: Tokenize - convert text to token IDs
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Step 2: Generate - model predicts next tokens
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=30  # Only generate 30 new tokens
)

# Step 3: Decode - convert token IDs back to text
print(tokenizer.decode(generation_output[0]))

## Understanding Tokenization: Looking Under the Hood

Let's examine what the tokenizer actually did. The `input_ids` tensor contains the numerical representation of our prompt.

In [ ]:
# Display the token IDs - these are the numbers the model actually sees
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


### Breaking Down the Tokenization

Let's decode each token ID individually to see how the text was split up. This reveals the tokenization strategy:

- Some tokens are whole words ("Write", "email")
- Some are word parts ("apolog", "izing")
- Some are punctuation (".", special tokens)

This is called **subword tokenization** - it balances vocabulary size with flexibility.

In [ ]:
# Decode each token ID one at a time
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


### The Complete Generation Process

Now let's look at the full output including the generated tokens:

In [ ]:
# The generation_output contains both input and generated token IDs
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799, 19235, 29892,    13,    13,    13, 29902,
          4966,   445,  2643, 14061]], device='cuda:0')

### Multi-Token Words

Some words get split into multiple tokens. Let's see an example with "Subject:"

In [ ]:
# Individual tokens that make up "Subject:"
print(tokenizer.decode(3323))  # "Sub"
print(tokenizer.decode(622))   # "ject"
print(tokenizer.decode([3323, 622]))  # "Subject" (combined)
print(tokenizer.decode(29901))  # ":"

Sub
ject
Subject
:


# Part 2: Comparing Different Tokenizers

## Why do different models tokenize differently?

Different language models use different tokenization strategies based on:
- **Training data**: What languages/text they were trained on
- **Vocabulary size**: Tradeoff between memory and granularity
- **Use case**: Code models tokenize differently than text models

Let's see how the same text gets tokenized by different models. The colored output helps visualize where token boundaries are.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Color codes for visualization (6 different colors)
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    """Display how a tokenizer splits text, with color-coding for each token"""
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids

    # Print each token with a different background color
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

### Test Text: A Mix of Challenges

We'll use text that includes various challenging elements:
- Regular English words
- CAPITALIZATION
- Emojis (🎵 🐦)
- Python code
- Numbers and operators

This shows how different tokenizers handle diverse content.

In [ ]:
text = """
English and CAPITALIZATION
🎵 🐦
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

### BERT (Cased): Preserves Capitalization

BERT uses **WordPiece tokenization** with these characteristics:
- Preserves case ("English" vs "english" are different)
- Uses `##` prefix for word continuations ("capital##ization")
- Has special tokens: `[CLS]` (start), `[SEP]` (end)
- Handles unknown characters as `[UNK]` (the emojis)

In [ ]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

### GPT-2: Byte-Level BPE

GPT-2 uses **Byte Pair Encoding (BPE)**:
- Can represent ANY text (including emojis) - no `[UNK]` tokens!
- Uses more tokens for the same text compared to BERT
- Notice how whitespace is explicitly tokenized
- Trained primarily on English web text

In [ ]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

### T5: SentencePiece Tokenization

T5 uses **SentencePiece**, which:
- Treats whitespace as a special character (▁)
- Purely data-driven (learns from training text)
- Can handle multiple languages
- Uses `<unk>` for unknown tokens instead of `[UNK]`

In [ ]:
show_tokens(text, "google/flan-t5-small")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 </s> 

### GPT-4: Modern Tokenization

GPT-4 tokenizer (via tiktoken):
- More efficient than GPT-2 - uses fewer tokens for same text
- Better handles code, numbers, and special characters
- Larger vocabulary = more direct mappings for common patterns

In [ ]:
# Using HuggingFace's version of the GPT-4 tokenizer
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 12 . 0 * 50 = 600 
 

### StarCoder: Code-Specialized Tokenization

StarCoder is trained on code, so:
- Recognizes programming keywords as single tokens
- Handles operators (==, >=) more efficiently
- Better at preserving code structure (indentation, syntax)

**Note**: You may need to request access to this model on HuggingFace first.

In [ ]:
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

### Phi-3: Our Demo Model

This is the same model we're using for generation. Notice how it tokenizes our test text.

In [ ]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

## Key Takeaway from Tokenizer Comparison

Different tokenizers make different tradeoffs:
- **Vocabulary size** vs **token sequence length**
- **Handling rare words** vs **memory efficiency**
- **Multilingual support** vs **English optimization**

There's no "best" tokenizer - it depends on your use case!

# Part 3: Contextualized Word Embeddings from BERT

## What are embeddings?

Embeddings convert tokens into dense numerical vectors (lists of numbers). But unlike simple lookup tables, **contextualized embeddings** change based on context.

### Example:
The word "bank" means different things in these sentences:
- "I deposited money at the **bank**" (financial institution)
- "We sat by the river **bank**" (land alongside water)

Contextualized embeddings give "bank" different numerical representations based on surrounding words!

## BERT: Bidirectional Encoder

BERT (Bidirectional Encoder Representations from Transformers) reads text in both directions:
- Looks at words **before** and **after** each token
- Creates embeddings that capture contextual meaning
- Each token gets a vector (list of numbers) representing its meaning in context

In [ ]:
from transformers import AutoModel, AutoTokenizer

# Load DeBERTa (an improved version of BERT)
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize our input
tokens = tokenizer('Hello world', return_tensors='pt')

# Get contextualized embeddings from the model
output = model(**tokens)[0]

### Understanding the Token Dictionary

The tokenizer returns more than just token IDs:

In [ ]:
print(tokens)
# input_ids: The token IDs themselves
# token_type_ids: Distinguishes different segments (used for sentence pairs)
# attention_mask: Indicates which tokens are real vs padding

{'input_ids': tensor([[    1, 31414,   232,     2]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}


### Shape of Embeddings: [batch, tokens, dimensions]

The output shape tells us:
- **1**: batch size (we processed 1 sentence)
- **4**: number of tokens ([CLS], "Hello", "world", [SEP])
- **384**: embedding dimension (each token → 384 numbers)

Each token is represented by a 384-dimensional vector!

In [ ]:
output.shape

torch.Size([1, 4, 384])

### The Token Breakdown

Let's see what tokens were created:

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

# [CLS] = classification token (marks beginning)
# Hello = first word
# world = second word (note the space is included!)
# [SEP] = separator token (marks end)

### The Actual Embedding Vectors

Here's what the embeddings actually look like - 384 numbers for each of our 4 tokens:

In [ ]:
output
# Notice: These are LEARNED representations
# The numbers encode semantic meaning based on BERT's training

# Part 4: Sentence-Level Text Embeddings

## From Token Embeddings to Sentence Embeddings

So far we've seen embeddings for individual tokens. But what if we want a single vector for an entire sentence or document?

### The Challenge:
- A sentence has many tokens → many embeddings
- We want: entire sentence → single embedding vector

### The Solution: Sentence Transformers

These are specially trained models that:
1. Process all tokens in the sentence
2. Combine them (usually via pooling) into one vector
3. Ensure similar sentences get similar embeddings

**Use cases**: Semantic search, finding similar documents, clustering, classification

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a sentence embedding model
# This model is trained to create meaningful sentence-level embeddings
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert an entire sentence to a single vector
vector = model.encode("Best movie ever!")

### Embedding Dimensions

Our sentence "Best movie ever!" is now represented as 768 numbers:

In [ ]:
vector.shape
# 768 dimensions - each number captures some aspect of the sentence's meaning
# Sentences with similar meanings will have similar vectors
# We can use cosine similarity to compare sentences!

(768,)

# Part 5: Classic Word Embeddings (Word2Vec, GloVe)

## Before Transformers: Static Word Embeddings

Before models like BERT, we had **static embeddings** where each word always gets the same vector, regardless of context.

### Word2Vec & GloVe:
- Each word → one fixed vector
- Captures semantic relationships ("king" - "man" + "woman" ≈ "queen")
- Much smaller and faster than modern transformers
- Still useful for many applications!

**Key Difference**: The word "bank" gets the same embedding whether it's a financial institution or a river bank. Context-free.

In [ ]:
# Install gensim if not already installed
!pip install gensim

In [ ]:
import gensim.downloader as api

# Download pre-trained embeddings (66MB)
# GloVe: Global Vectors for Word Representation
# Trained on Wikipedia, 50-dimensional vectors
model = api.load("glove-wiki-gigaword-50")

# Alternative: model = api.load("word2vec-google-news-300")
# More options at: https://github.com/RaRe-Technologies/gensim-data

### Exploring Semantic Similarity

Let's find words similar to "king". The model compares embedding vectors and returns words with similar vectors:

In [ ]:
model.most_similar([model['king']], topn=11)

# Results show:
# - Word similarities are based on how words are used in text
# - "prince", "queen", "emperor" are all related to royalty
# - Numbers show cosine similarity (1.0 = identical, 0.0 = unrelated)
# - "king" is most similar to itself (1.0)

# Part 6: Real Application - Music Recommendation System

## How can we use embeddings for recommendations?

Here's a practical example using Word2Vec on music playlists:

### The Idea:
1. Treat each song as a "word"
2. Treat playlists as "sentences"
3. Songs that appear together in playlists → similar embeddings
4. Use embeddings to find similar songs!

This is the same technique used by:
- Spotify (music recommendations)
- Amazon (product recommendations)  
- YouTube (video recommendations)

---

## Why ID-Based Embeddings vs. Content-Based Embeddings?

### Understanding the Two Approaches

**ID-Based (Collaborative Filtering) Embeddings:**
- Each song gets a unique ID (like "song_2172")
- Embeddings are learned from **user behavior patterns**
- Songs that appear together in playlists get similar embeddings
- Also called "collaborative filtering" or "behavioral embeddings"

**Content-Based Embeddings:**
- Analyze the **actual content** of songs: audio features, lyrics, metadata
- Use audio analysis (tempo, key, timbre, energy) from spectrograms
- Process lyrics with NLP models
- Based on what the song "sounds like" or "is about"

---

### When ID-Based Embeddings Excel (Usually!)

**1. Captures the "Hidden Dimensions" of Human Taste**

Behavioral embeddings learn patterns that acoustic analysis fundamentally cannot capture:
- **Cultural context**: Two songs might sound different but both be "90s nostalgia"
- **Activity associations**: Songs for "workout" vs "study" vs "party"
- **Mood and vibe**: The ineffable quality that makes songs "feel right" together
- **Social trends**: What's popular in specific communities or demographics

*Example*: A heavy metal song and a classical piece might both appear in "Epic Movie Soundtrack" playlists. Behavioral embeddings capture this; audio analysis would mark them as completely dissimilar.

**2. Learns from Curated Human Judgment**

Spotify processes **700 million user-generated playlists**. Each playlist represents a human's intentional curation - their statement about what songs belong together. This signal is incredibly powerful:
- Playlist co-occurrence is stronger than just co-listening
- Users put effort into organizing playlists by theme, mood, activity
- Aggregating millions of playlists reveals consensus about similarity

**3. Empirical Performance**

Research shows collaborative filtering typically achieves:
- **4-8% improvement in Recall@20** over content-only methods
- **Better long-term engagement**: users stay on platform longer
- **Higher satisfaction**: recommendations feel more "right"

---

### When Content-Based Embeddings Win

**1. The Cold Start Problem**

This is where content-based approaches are **essential**:
- **New songs**: Zero listening history → behavioral embeddings can't help
- **New users**: No interaction data → can't use collaborative filtering
- **New platforms**: Launching in new markets with no local user data

*Solution*: Content features (audio analysis, lyrics) provide immediate recommendations

**2. Discovering Niche and Long-Tail Content**

Collaborative filtering has a **popularity bias**:
- Just **5% of artists account for 62%+ of interactions**
- Obscure artists rarely get recommended (no co-occurrence data)
- Creates a "rich get richer" effect

Content-based approaches:
- Link artists based on actual sound, not popularity
- Enable discovery of niche music in 2-3 steps from mainstream
- Correlation with popularity: **r=0.10** (vs r=0.38 for CF)

**3. Explainability and Transparency**

Content-based systems can explain *why* songs are similar:
- "Both have 120 BPM and high energy"
- "Similar guitar-driven indie rock sound"
- "Both songs discuss heartbreak themes"

Behavioral systems can only say: "Users like you enjoyed this"

---

### What the Industry Actually Does: Hybrid Systems

**No major company uses just one approach!** All production systems combine both:

**Spotify's Three-Pronged Approach:**
1. Collaborative filtering on playlist co-occurrence (our approach in this lab!)
2. Audio analysis using CNNs on spectrograms (42+ feature dimensions)
3. NLP on lyrics and artist descriptions

Result: **16 billion artist discoveries per month**

**YouTube's Two-Tower Architecture:**
- User tower: encodes watch history, demographics, context
- Video tower: encodes video ID + content features + metadata
- Both produce embeddings, similarity via dot product
- Handles both warm (lots of views) and cold (new uploads) videos

**Netflix:**
- **80%+ of watched content** comes from recommendations
- Combines: collaborative filtering, matrix factorization, 250+ content features
- Graph neural networks linking movies, genres, actors, directors
- Saves **$1+ billion annually** by reducing churn

---

### Bottom Line for This Lab

**Why we use ID-based embeddings here:**
- ✅ We have rich playlist data (behavioral signals)
- ✅ We want to learn what songs "go together" from user curation
- ✅ Simple to implement with Word2Vec
- ✅ Demonstrates collaborative filtering principles

**When you'd add content-based embeddings:**
- ❌ Recommending brand new songs (cold start)
- ❌ Discovering obscure artists (long-tail problem)
- ❌ Explaining why recommendations make sense
- ❌ Launching in new markets with no user data

**The Professional Approach:**
In a real production system, you'd use **BOTH**:
- Start with content features for new items
- Transition to behavioral embeddings as data accumulates  
- Blend both in a two-tower neural network
- Continuously measure performance on cold vs. warm items separately

**Key Insight**: *Behavioral embeddings capture what songs mean to people; content embeddings capture what songs sound like. Both are valuable, but for different reasons!*

### Loading the Playlist Dataset

We're using the Yes.com dataset:
- Contains thousands of real user playlists
- Each playlist is a sequence of song IDs
- We also load song metadata (titles and artists)

In [ ]:
import pandas as pd
from urllib import request

# Download playlist data
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse playlists (skip first 2 lines - they're metadata)
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song (can't learn patterns from those!)
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata (ID → Title, Artist)
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

### What Do Playlists Look Like?

Each playlist is just a list of song IDs. Songs that appear together frequently will learn similar embeddings.

In [ ]:
print('Playlist #1:\n ', playlists[0], '\n')
print('Playlist #2:\n ', playlists[1])

# Notice: Song IDs repeat across playlists
# Songs appearing together → learn similar embeddings

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

## Training Word2Vec on Playlists

Now we train our own Word2Vec model on these playlists:

### Parameters Explained:
- **vector_size=32**: Each song → 32-dimensional vector (smaller = faster, less nuanced)
- **window=20**: Look at 20 songs before/after for context (larger window = more context)
- **negative=50**: Number of "negative samples" for training (improves quality)
- **min_count=1**: Keep all songs, even if they appear only once
- **workers=4**: Use 4 cores for parallel training

**Training time**: Usually a few seconds on this dataset

In [ ]:
from gensim.models import Word2Vec

# Train Word2Vec on our playlists
model = Word2Vec(
    playlists,
    vector_size=32,    # Dimensionality of embeddings
    window=20,         # Context window size
    negative=50,       # Negative sampling parameter
    min_count=1,       # Minimum song occurrences to include
    workers=4          # Parallel processing
)

### Finding Similar Songs

Now we can ask: "What songs are similar to song #2172?"

The model finds songs with similar embeddings (i.e., songs that appear in similar playlist contexts):

In [ ]:
song_id = 2172

# Find the 10 most similar songs based on embedding similarity
model.wv.most_similar(positive=str(song_id))

# Returns: (song_id, similarity_score) pairs
# Higher score = more similar embedding = more likely to be enjoyed together

### What Song is #2172?

Let's see what song we're getting recommendations for:

In [ ]:
print(songs_df.iloc[2172])
# This tells us the title and artist

## Building a Recommendation Function

Let's make this more user-friendly by creating a function that shows song names instead of just IDs:

In [ ]:
import numpy as np

def print_recommendations(song_id):
    """Get top 5 song recommendations based on embedding similarity"""

    # Get similar song IDs and their scores
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=5)
    )[:,0]  # Extract just the song IDs (ignore scores)

    # Look up song information and return as DataFrame
    return songs_df.iloc[similar_songs]

# Test it!
print_recommendations(2171)

### Analyzing the Results

Look at the recommendations! If song 2172 is "Fade To Black" by Metallica, the recommendations are likely other heavy metal songs from the same era (Metallica, Iron Maiden, Judas Priest, etc.).

**Why does this work?**
- People who like Metallica often create playlists with similar bands
- These songs appear together frequently in playlists
- Word2Vec learns these patterns → similar embeddings

In [ ]:
# Try another example
print_recommendations(2172)

### Test with Different Genres

Let's try a different song (ID 842) and see if we get different genre recommendations:

In [ ]:
print_recommendations(842)

# Notice: If this is a hip-hop/rap song, recommendations will be similar genre
# The embeddings captured genre patterns from playlist co-occurrence!

## Summary: What We Have Learned

### 1. Tokenization
- Breaks text into processable units (tokens)
- Different strategies: word-level, subword (BPE, WordPiece), character-level
- Choice affects model performance and vocabulary size

### 2. Token Embeddings (Contextualized)
- BERT-style models create different embeddings based on context
- Same word in different contexts → different vectors
- Shape: [batch_size, num_tokens, embedding_dim]

### 3. Sentence Embeddings
- Combine token embeddings → single vector per sentence
- Used for semantic similarity, search, clustering
- Models like Sentence-BERT are specifically trained for this

### 4. Classic Word Embeddings
- Word2Vec, GloVe: one vector per word (context-independent)
- Smaller, faster, still useful for many tasks
- Can be trained on any sequential data (words in sentences, songs in playlists, etc.)

### The Big Picture
All modern language models rely on these fundamental concepts:
1. **Tokenize** text into processable units
2. **Embed** tokens into numerical space
3. **Process** with neural networks to make predictions
4. **Decode** back to human-readable text

You've now seen each step in action! 🎉

---

# 🎯 Fun Exercise: Build Your Own Movie Recommendation System

## Goal
Build a complete movie recommendation system that compares **collaborative filtering** (ID-based) vs **content-based** approaches, implements proper evaluation metrics, and handles cold-start scenarios.

**Time estimate**: ~2 hours for average graduate students

---

## Setup: Download MovieLens Dataset

We'll use the MovieLens-100k dataset:
- 100,000 ratings from 943 users on 1,682 movies
- Movie metadata (titles, genres, release dates)
- User demographics (age, gender, occupation)

```python
# Download and extract MovieLens-100k
!wget http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip
```

---

## Part A: Collaborative Filtering with User Ratings (40 minutes)

### Task 1: Build Rating-Based Sequences

**Your mission**: Create "user sequences" where each user's sequence is their rated movies in chronological order.

**Steps**:
1. Load `u.data` (user_id, item_id, rating, timestamp)
2. Filter to only highly-rated movies (rating >= 4) - why do we do this?
3. Sort by user_id and timestamp
4. Create sequences: each user → list of movie IDs they rated highly

**Starter code**:
```python
import pandas as pd

# Load ratings
ratings = pd.read_csv('ml-100k/u.data', sep='\t',
                      names=['user_id', 'item_id', 'rating', 'timestamp'])

# TODO: Your code here
# Hint: Use groupby and apply to create sequences per user
```

**Expected output**: A list of lists, where `user_sequences[0]` might look like `['242', '393', '381', ...]`

---

### Task 2: Train Word2Vec on User Rating Sequences

**Your mission**: Train a Word2Vec model treating movies like words and user rating sequences like sentences.

**Decisions you need to make**:
- `vector_size`: How many dimensions? (Try 64, 128, or 256)
- `window`: How many movies before/after to consider? (Try 5, 10, or 20)
- `min_count`: Minimum movie appearances? (Try 5 or 10 to filter rare movies)

**Think about**: Why might movie recommendations need different hyperparameters than song recommendations?

---

### Task 3: Implement and Test Recommendations

**Your mission**: Create a function that recommends movies and displays their titles.

```python
# Load movie metadata
movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1',
                     names=['movie_id', 'title', 'release_date', 'video_release_date',
                            'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation',
                            'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
                            'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
                            'Thriller', 'War', 'Western'])

# TODO: Implement get_movie_recommendations(movie_id, top_n=5)
```

**Test with**:
- Movie ID 1: "Toy Story (1995)"
- Movie ID 50: "Star Wars (1977)"
- Movie ID 100: "Fargo (1996)"

**Question**: Do the recommendations make sense? Why or why not?

---

## Part B: Content-Based Filtering with Genre Features (40 minutes)

### Task 4: Build Content-Based Embeddings

**Your mission**: Create embeddings based on movie genres (columns 6-23 in u.item).

**Approaches to try**:

**Option 1: Simple Genre Vectors**
- Use the binary genre columns directly as embeddings
- Each movie → 19-dimensional binary vector

**Option 2: TF-IDF on Genre Combinations**
- Treat genre combinations as "documents"
- Use TfidfVectorizer from sklearn

**Option 3 (Advanced): Train Doc2Vec on Movie Descriptions**
- Scrape or use pre-existing movie plot summaries
- Train Doc2Vec to get content-based embeddings

**Starter code**:
```python
from sklearn.metrics.pairwise import cosine_similarity

# Extract genre features (columns 6-23)
genre_features = movies.iloc[:, 6:24].values

# TODO: Implement content-based similarity
# Hint: Use cosine_similarity to find similar movies
```

**Test**: Find movies similar to "Toy Story" based on genres. Compare to collaborative filtering results.

---

## Part C: Quantitative Evaluation (30 minutes)

### Task 5: Implement Evaluation Metrics

**Your mission**: Properly evaluate both approaches using train/test splits.

**Metrics to implement**:

1. **Precision@K**: Of the top K recommendations, what % did the user actually rate highly?
   ```
   Precision@5 = (# of relevant items in top 5) / 5
   ```

2. **Recall@K**: Of all movies the user liked, what % are in top K recommendations?
   ```
   Recall@5 = (# of relevant items in top 5) / (total # of relevant items)
   ```

3. **NDCG@K (Normalized Discounted Cumulative Gain)**: Accounts for ranking position
   - Use `from sklearn.metrics import ndcg_score`

**Evaluation protocol**:
```python
# 1. Split data: 80% train, 20% test (chronologically)
# 2. Train model on training sequences only
# 3. For each user in test set:
#    - Get their test movies (ground truth)
#    - Generate top-K recommendations
#    - Calculate metrics
# 4. Average metrics across all test users
```

**Challenge question**: Why is chronological splitting important? What happens if you split randomly?

---

### Task 6: Compare Collaborative vs Content-Based

**Your mission**: Create a comparison table of performance metrics.

**Expected output**:
```
| Approach              | Precision@5 | Recall@10 | NDCG@10 |
|-----------------------|-------------|-----------|----------|
| Collaborative (CF)    | 0.XX        | 0.XX      | 0.XX     |
| Content-Based         | 0.XX        | 0.XX      | 0.XX     |
| Hybrid (Average)      | 0.XX        | 0.XX      | 0.XX     |
```

**Analysis questions**:
1. Which approach performs better overall? By how much?
2. Are there specific movie genres where one approach excels?
3. What about popular vs. obscure movies?

---

## Part D: Cold-Start Challenge (20 minutes)

### Task 7: Simulate Cold-Start Scenario

**Your mission**: Test what happens with brand new movies (no rating history).

**Experiment design**:
1. Select 20 random movies from your dataset
2. **Remove all their ratings** from training data (simulate they're brand new)
3. Try to recommend these movies using:
   - Collaborative filtering (will it work? why/why not?)
   - Content-based approach (should work!)
4. Compare to ground truth (the ratings you removed)

**Expected insight**: You should see collaborative filtering completely fail for these movies, while content-based can still make recommendations.

**Bonus**: Implement a hybrid approach that:
- Uses content-based for cold items (< 5 ratings)
- Uses collaborative for warm items (>= 5 ratings)


---

## Part E: Hybrid System (Bonus - 10 minutes)

### Task 8: Build a Simple Hybrid Recommender

**Your mission**: Combine collaborative and content-based scores.

**Approach**:
```python
def hybrid_recommendations(movie_id, alpha=0.7):
    """
    alpha: weight for collaborative filtering (0 to 1)
    (1-alpha): weight for content-based
    """
    # Get CF scores
    cf_scores = get_collaborative_scores(movie_id)
    
    # Get content scores
    content_scores = get_content_scores(movie_id)
    
    # Normalize both to [0, 1] range
    # Combine: alpha * cf_scores + (1-alpha) * content_scores
    # Return top K
```

**Experiment**: Try different alpha values (0.3, 0.5, 0.7, 0.9) and measure performance. What's optimal?

---

## Deliverables

Your completed exercise should include:

### 1. Code Implementation (Required)
- ✅ Collaborative filtering recommender with Word2Vec
- ✅ Content-based recommender using genres
- ✅ Evaluation metrics (Precision@K, Recall@K, NDCG@K)
- ✅ Cold-start simulation
- ✅ At least one visualization

### 2. Analysis Report (Required)

Answer these questions in markdown cells:

**A. Performance Comparison**
- Which approach achieved better metrics? By how much?
- Were there specific genres or movie types where one approach excelled?
- What hyperparameters did you try? Which worked best?

**B. Cold-Start Analysis**
- What happened to collaborative filtering on cold-start movies?
- How much worse was content-based on warm items vs CF?
- Would a hybrid approach solve this? Show results.

**C. Real-World Application**
- If you were building a movie streaming service, which approach would you use?
- How would you handle the first 100 users? First 1000 movies?
- What data would you prioritize collecting?

### 3. Bonus Challenges (Optional)

**Implement any of these for extra insight**:

- **Diversity metrics**: Do recommendations span multiple genres or get stuck in one?
- **Temporal analysis**: Do older vs newer movies behave differently?
- **User segmentation**: Do different user demographics need different approaches?
- **Implicit feedback**: Use watch time / completion rate instead of explicit ratings
- **Two-tower model**: Implement a simple neural two-tower architecture
- **A/B test simulation**: Randomly assign users to CF vs content-based, compare engagement

---

## Hints and Tips

### Common Pitfalls

1. **Don't forget to convert movie IDs to strings** for Word2Vec (it expects strings)
2. **Handle movies with no embeddings** (not enough occurrences) - use try/except
3. **Normalize scores** before combining in hybrid approach
4. **Use chronological splits** for realistic evaluation (simulate real deployment)

### Debugging Tips

- Print sequence lengths - are they reasonable? (should be 5-50 movies per user)
- Check vocabulary size - did Word2Vec include most movies?
- Verify metrics are between 0 and 1 - if not, check your calculation
- Sample a few recommendations manually - do they make sense?

### Expected Performance Ranges

Based on literature, you should see approximately:
- **Collaborative Filtering**: Precision@5: 0.20-0.35, NDCG@10: 0.25-0.40
- **Content-Based (genres only)**: Precision@5: 0.15-0.25, NDCG@10: 0.18-0.30
- **Hybrid**: Precision@5: 0.22-0.37, NDCG@10: 0.27-0.42

If your numbers are very different, debug your implementation!

---

## Grading Criteria (Self-Assessment)

### Basic (70%)
- ✅ Collaborative filtering recommender works correctly
- ✅ Content-based recommender implemented
- ✅ At least Precision@K and Recall@K calculated
- ✅ Comparison table showing performance differences
- ✅ Basic analysis answering key questions

### Proficient (85%)
- ✅ All basic requirements
- ✅ NDCG metric correctly implemented
- ✅ Cold-start simulation showing clear performance difference
- ✅ Thoughtful analysis of when to use each approach

### Excellent (100%)
- ✅ All proficient requirements
- ✅ Hybrid approach implemented and evaluated
- ✅ Multiple hyperparameter experiments with clear methodology
- ✅ Deep analysis: specific examples, edge cases, failure modes
- ✅ One or more bonus challenges completed
- ✅ Production considerations discussed (scalability, real-time, etc.)

---

## Additional Resources

**Documentation**:
- Gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html
- MovieLens dataset: https://grouplens.org/datasets/movielens/

**Research papers** (if you want to go deeper):
- Item2Vec (applying Word2Vec to recommendations): https://arxiv.org/vc/arxiv/papers/1603/1603.04259v2.pdf
- YouTube's Deep Neural Networks for Recommendations: https://cseweb.ucsd.edu/classes/fa17/cse291-b/reading/p191-covington.pdf
- Netflix's matrix factorization approach: https://datajobs.com/data-science-repo/Recommender-Systems-%5BNetflix%5D.pdf

**Pro tip**: Check out Spotify's engineering blog for real-world insights on production recommendation systems!

---

## Getting Started

Create new cells below and start with Part A, Task 1. Work through systematically, testing each component before moving to the next. Good luck!